# DML na camada Bronze (Delta Lake no MinIO)

Este notebook executa **INSERT**, **UPDATE** e **DELETE** sobre uma tabela Delta já materializada no bucket **`bronze`** (objeto compatível com S3 via **s3a**), e em seguida consulta o **histórico de transações** do Delta para evidenciar rastreabilidade.

**Pré-requisito:** pipeline até a Bronze concluído (por exemplo `notebook/02_landing_to_bronze_delta.ipynb`), MinIO acessível e variáveis de ambiente carregadas (ou credenciais padrão de desenvolvimento). O bucket **`bronze`** deve existir (criado pelo `minio-init`, pelo notebook **`02`** ou manualmente na consola MinIO).

## 1. Credenciais MinIO e caminho da tabela

Os valores vêm do `.env` na raiz do projeto quando existirem; caso contrário, usam-se **`minioadmin`** / **`minioadmin`** e o endpoint **`http://localhost:9000`**, conforme convenção típica do MinIO local.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

_root = Path.cwd()
if _root.name == "notebook":
    _root = _root.parent
load_dotenv(_root / ".env")

MINIO_ENDPOINT = os.getenv("MINIO_S3_ENDPOINT", "http://localhost:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

BRONZE_BUCKET = "bronze"
TABELA_BRONZE = "dbo_Categoria"
BRONZE_TABLE_PATH = f"s3a://{BRONZE_BUCKET}/{TABELA_BRONZE}/"

print("MinIO endpoint:", MINIO_ENDPOINT)
print("Tabela Delta:", BRONZE_TABLE_PATH)

MinIO endpoint: http://127.0.0.1:9000
Tabela Delta: s3a://bronze/dbo_Categoria/


## 2. SparkSession com Delta Lake e S3A (MinIO)

O Spark carrega os JARs do **Delta**, **Hadoop AWS** e **AWS SDK** para falar com o MinIO pelo protocolo **s3a**, com acesso por chave estilo path (`path.style.access`).

In [2]:
from pyspark.sql import SparkSession

packages = (
    "io.delta:delta-spark_2.12:3.2.0,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
)

spark = (
    SparkSession.builder.appName("dml-bronze-delta-minio")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.jars.packages", packages)
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

26/05/06 20:34:29 WARN Utils: Your hostname, PC-Octavio resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/06 20:34:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/source/trabalho2ed-minio-sql/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/octavio/.ivy2/cache
The jars for the packages stored in: /home/octavio/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-07621f4f-02d7-43d3-a0d9-ce3dda6815ca;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 287ms :: artifacts dl 5ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4

Spark: 3.5.3


## 3. Leitura da tabela Bronze

Foi escolhida a tabela **`dbo_Categoria`** (extraída do SQL Server e gravada como Delta no notebook `02`), por ter chave simples (`id_categoria`) e colunas adequadas para demonstrar mutação e expurgo.

In [3]:
import pyspark.sql.functions as F

df_bronze = spark.read.format("delta").load(BRONZE_TABLE_PATH)
total_inicial = df_bronze.count()
print("Registros antes do DML:", total_inicial)
df_bronze.orderBy("id_categoria").show(10, truncate=False)

26/05/06 20:34:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/06 20:34:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Registros antes do DML: 5


+------------+-----------------+------------------------------------------------------------------------------------+--------------------------+-----------------------------------------------------------------------------------------+
|id_categoria|nome             |descricao                                                                           |_bronze_loaded_at         |_bronze_source_file                                                                      |
+------------+-----------------+------------------------------------------------------------------------------------+--------------------------+-----------------------------------------------------------------------------------------+
|0           |Romance          |Ficção narrativa centrada em relacionamentos sentimentais e conflitos humanos.      |2026-05-06T20:33:35.445810|s3a://landing-zone/dbo_Categoria/part-00000-a2e5bf34-7b1e-4487-809e-66f17c053005-c000.csv|
|1           |Ficção Científica|Narrativas com tecnologia fu

## 4. INSERT — registro fictício

Inserimos uma **nova categoria** com `id_categoria` derivado de `max(id_categoria) + 1`, preenchendo também os metadados de auditoria da Bronze com `pyspark.sql.functions` (`F.lit`, `F.current_timestamp`).

In [4]:
df_atual = spark.read.format("delta").load(BRONZE_TABLE_PATH)
proximo_id = df_atual.select(F.max(F.col("id_categoria")).alias("m")).first()["m"]
base_id = int(proximo_id) if proximo_id is not None else 0
novo_id = base_id + 1

linha_nova = spark.range(1).select(
    F.lit(novo_id).cast("int").alias("id_categoria"),
    F.lit("Categoria fictícia (INSERT DML)").alias("nome"),
    F.lit("Registro sintético para validar escrita ACID na Bronze.").alias("descricao"),
    F.date_format(
        F.current_timestamp(), "yyyy-MM-dd'T'HH:mm:ss.SSSSSS"
    ).alias("_bronze_loaded_at"),
    F.lit("dml_bronze.ipynb#INSERT").alias("_bronze_source_file"),
)

linha_nova.write.format("delta").mode("append").save(BRONZE_TABLE_PATH)

spark.read.format("delta").load(BRONZE_TABLE_PATH).filter(
    F.col("id_categoria") == F.lit(novo_id)
).show(truncate=False)
print("id inserido (fictício):", novo_id)

+------------+-------------------------------+-------------------------------------------------------+--------------------------+-----------------------+
|id_categoria|nome                           |descricao                                              |_bronze_loaded_at         |_bronze_source_file    |
+------------+-------------------------------+-------------------------------------------------------+--------------------------+-----------------------+
|5           |Categoria fictícia (INSERT DML)|Registro sintético para validar escrita ACID na Bronze.|2026-05-06T20:34:52.977037|dml_bronze.ipynb#INSERT|
+------------+-------------------------------+-------------------------------------------------------+--------------------------+-----------------------+

id inserido (fictício): 5


## 5. UPDATE — correção pontual

Atualizamos a coluna **`descricao`** de um registro existente (`id_categoria = 1`) via API do Delta, sem regravar o bucket inteiro manualmente.

In [5]:
from delta.tables import DeltaTable

tabela_delta = DeltaTable.forPath(spark, BRONZE_TABLE_PATH)

tabela_delta.update(
    condition="id_categoria = 1",
    set={
        "descricao": F.lit(
            "Descrição revisada na Bronze (UPDATE via DeltaTable)."
        )
    },
)

spark.read.format("delta").load(BRONZE_TABLE_PATH).filter(
    F.col("id_categoria") == F.lit(1)
).select("id_categoria", "nome", "descricao").show(truncate=False)

+------------+-----------------+-----------------------------------------------------+
|id_categoria|nome             |descricao                                            |
+------------+-----------------+-----------------------------------------------------+
|1           |Ficção Científica|Descrição revisada na Bronze (UPDATE via DeltaTable).|
+------------+-----------------+-----------------------------------------------------+



## 6. DELETE — expurgo do registro fictício

Removemos apenas o registro inserido no passo de **INSERT**, simulando **expurgo** (por exemplo dado de teste ou linha que não deve permanecer na camada curada).

In [6]:
tabela_delta = DeltaTable.forPath(spark, BRONZE_TABLE_PATH)
tabela_delta.delete(F.col("id_categoria") == F.lit(novo_id))

df_pos = spark.read.format("delta").load(BRONZE_TABLE_PATH)
print("Registros após DELETE:", df_pos.count())
df_pos.filter(F.col("id_categoria") == F.lit(novo_id)).count()
df_pos.orderBy("id_categoria").show(8, truncate=False)

Registros após DELETE: 5
+------------+-----------------+------------------------------------------------------------------------------------+--------------------------+-----------------------------------------------------------------------------------------+
|id_categoria|nome             |descricao                                                                           |_bronze_loaded_at         |_bronze_source_file                                                                      |
+------------+-----------------+------------------------------------------------------------------------------------+--------------------------+-----------------------------------------------------------------------------------------+
|0           |Romance          |Ficção narrativa centrada em relacionamentos sentimentais e conflitos humanos.      |2026-05-06T20:33:35.445810|s3a://landing-zone/dbo_Categoria/part-00000-a2e5bf34-7b1e-4487-809e-66f17c053005-c000.csv|
|1           |Ficção Científica|Des

## 7. Auditoria — `DeltaTable.forPath` e `history()`

O método **`history()`** expõe cada transação (versão, carimbo de tempo, operação e parâmetros), permitindo relacionar **INSERT**, **UPDATE** e **DELETE** a commits consecutivos no log do Delta armazenado junto aos dados no MinIO.

In [7]:
tabela_delta = DeltaTable.forPath(spark, BRONZE_TABLE_PATH)

tabela_delta.history().select(
    "version",
    "timestamp",
    "operation",
    "operationParameters",
).orderBy(F.col("version").desc()).show(20, truncate=80)

+-------+-------------------+---------+------------------------------------------+
|version|          timestamp|operation|                       operationParameters|
+-------+-------------------+---------+------------------------------------------+
|      3|2026-05-06 20:34:59|   DELETE|{predicate -> ["(id_categoria#3364 = 5)"]}|
|      2|2026-05-06 20:34:57|   UPDATE|{predicate -> ["(id_categoria#2025 = 1)"]}|
|      1|2026-05-06 20:34:54|    WRITE|       {mode -> Append, partitionBy -> []}|
|      0|2026-05-06 20:33:41|    WRITE|    {mode -> Overwrite, partitionBy -> []}|
+-------+-------------------+---------+------------------------------------------+



## 8. Delta Lake no MinIO versus banco relacional tradicional

| Aspecto | Delta no objeto (MinIO / S3A) | SGBDR comum |
|---------|-------------------------------|-------------|
| **Armazenamento** | Arquivos de dados + log transacional (`_delta_log`) no mesmo *prefixo* da tabela | Páginas e logs internos do motor, em geral opacos ao consumidor analítico |
| **Auditoria de mudanças** | `history()` lista versões e operações de forma explícita para engenharia de dados | Depende de triggers, tabelas de auditoria, CDC licenciado ou ferramentas externas |
| **Reprodutibilidade** | *Time travel* por versão ou timestamp facilita reexecutar análises sobre estado anterior | *Point-in-time* exige backup/restauração ou réplicas específicas |
| **Escopo típico** | *Data lake* / medalhão com Spark e grandes volumes | Transações OLTP com modelo relacional forte |

Em resumo: no **Delta sobre MinIO**, a rastreabilidade das DML fica **materializada no próprio conjunto de objetos** da tabela, consultável com APIs abertas do Spark. Em um **banco relacional**, a mesma visão costuma exigir **instrumentação adicional** ou produtos pagos, e nem sempre fica alinhada ao mesmo modelo de dados analíticos que rodam no lake.